# 05 — Debugging Policy Violations

How to inspect denials, validate policies, and use the AGT CLI.

In [ ]:
import sys; sys.path.insert(0, '..')
from pathlib import Path
import json

# Generate some audit entries by running a few scenarios
from src.agents.customer_service import CustomerServiceAgent
from src.models import AgentRole

cs = CustomerServiceAgent(role=AgentRole.CASHIER)
cs.invoke_tool('access_customer_pii', {'customer_id': 'CUST-001'})  # denied

In [ ]:
# Read audit log
log_path = Path('logs/governance-audit.jsonl')
if log_path.exists():
    entries = [json.loads(l) for l in log_path.read_text().strip().splitlines()]
    for e in entries[-5:]:
        status = 'ALLOW' if e['allow'] else 'DENY'
        print(f"[{status}] {e['policy_path']} | reason={e.get('reason')} | agent={e['agent_id']}")
else:
    print('No audit log yet — run some scenarios first')

## AGT CLI Tools

In [ ]:
import subprocess
result = subprocess.run(['agt', 'lint-policy', 'src/policies/'], capture_output=True, text=True)
print(result.stdout or result.stderr or 'Lint passed')

In [ ]:
result = subprocess.run(['agt', 'verify'], capture_output=True, text=True)
print(result.stdout or result.stderr or 'Verify passed')

## Denial Structure

Every denial returns:
```json
{"ok": false, "denied_by": "AGT-A", "reason": "Cashiers cannot access customer PII"}
```

- `denied_by`: phase that denied (`AGT-A` or `AGT-B`)
- `reason`: human-readable description from the policy rule

In [ ]:
# Inspect denial in detail
from src.agents.order_processing import OrderProcessingAgent

op = OrderProcessingAgent(role=AgentRole.CASHIER)
result = op.invoke_tool('process_order', {'customer_id': 'CUST-001', 'amount': 600.0})
print(json.dumps(result, indent=2))